# Imports


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import struct  # type: ignore[reportUnusedImport]
import sys  # type: ignore[reportUnusedImport]
import zipfile  # type: ignore[reportUnusedImport]
from collections import namedtuple  # type: ignore[reportUnusedImport]
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor  # type: ignore[reportUnusedImport]
from dataclasses import dataclass, field  # type: ignore[reportUnusedImport]
from datetime import datetime  # type: ignore[reportUnusedImport]
from functools import partial  # type: ignore[reportUnusedImport]
from multiprocessing import Pool  # type: ignore[reportUnusedImport]
from pathlib import Path  # type: ignore[reportUnusedImport]
from typing import List, Optional  # type: ignore[reportUnusedImport]

import geopandas as gpd  # type: ignore[reportUnusedImport]
import ggpymanager as ggp
import hvplot.xarray  # type: ignore[reportUnusedImport]
import matplotlib as mpl  # type: ignore[reportUnusedImport]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydeck as pdk  # type: ignore[reportUnusedImport]
import pyproj  # type: ignore[reportUnusedImport]
import rioxarray  # type: ignore[reportUnusedImport]
import rioxarray as rio  # type: ignore[reportUnusedImport]
import shapely  # type: ignore[reportUnusedImport]
import xarray as xr
from compliance_checker.runner import CheckSuite, ComplianceChecker  # type: ignore[reportUnusedImport]
from dask.diagnostics.progress import ProgressBar  # type: ignore[reportUnusedImport]
from joblib import Parallel, delayed  # type: ignore[reportUnusedImport]
from mpl_toolkits.axes_grid1.inset_locator import inset_axes  # type: ignore[reportUnusedImport]
from rasterio.enums import Resampling  # type: ignore[reportUnusedImport]
from tqdm import tqdm  # type: ignore[reportUnusedImport]
from windrose import WindroseAxes  # type: ignore[reportUnusedImport]

import paris_2025 as p  # type: ignore[reportUnusedImport]
from paris_2025.config import CONFIG
from paris_2025.plotting import RC_PARAMS

Using tracer path: /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers


In [3]:
# Get the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

plt.rcParams.update(RC_PARAMS)

# Fluxes


In [4]:
cadastre_path: str | Path = Path(CONFIG["domain"]["gral"]["conf_path"]) / "cadastre.dat"
source_groups_path: str | Path = p.model_input.fluxes.SOURCE_GROUP_NETCDF_PATH
point_path: str | Path = Path(CONFIG["domain"]["gral"]["conf_path"]) / "point.dat"
cadastre_emissions, source_groups, point_da, GRAL = (
    p.plotting._loaders.load_flux_maps_data(
        cadastre_path, source_groups_path, point_path
    )
)

In [5]:
point_fluxes = point_da.groupby("type").sum()
area_fluxes = cadastre_emissions.sum(["x", "y"]).groupby("type").sum()
df = pd.concat(
    [area_fluxes.to_pandas(), point_fluxes.to_pandas()],  # type: ignore
    axis=1,
    keys=["area", "point"],
)
# Convert from kg/h to kt/year
df = df * 365 * 24 / 1e6
df.round(0)

,area,point
type,,
Origins.earth 2023 energie,308.0,751.0
Origins.earth 2023 industrie,582.0,60.0
Origins.earth 2023 residentiel,1894.0,NaN
Origins.earth 2023 respiration_humaine,965.0,NaN
Origins.earth 2023 tertiaire,466.0,NaN
Origins.earth 2023 transport_routier,2707.0,NaN
TNO 2018 Combustion,4696.0,NaN
TNO 2018 Industry,205.0,NaN
TNO 2018 Power,135.0,2481.0


In [6]:
inventories = ["Origins.earth", "TNO"]
totals = pd.DataFrame(
    {inv: df[df.index.str.contains(inv)].sum() for inv in inventories}
).T
totals["total"] = totals.sum(axis=1)

relative_contributions = totals.div(totals["total"], axis=0) * 100

display(totals.round(0))
display(relative_contributions.round(1))

,area,point,total
Origins.earth,6921.0,811.0,7732.0
TNO,6950.0,2481.0,9431.0


,area,point,total
Origins.earth,89.5,10.5,100.0
TNO,73.7,26.3,100.0


# CO2 Measurements


In [7]:
def create_analysis_table(model, measurements, background):
    diff = model - measurements
    diff_background = background - measurements
    mean_enhancement = diff_background.mean("time")
    number_of_points = measurements.notnull().sum("time")
    mae = (np.abs(diff)).mean("time")
    rmse = np.sqrt(((diff) ** 2).mean("time"))
    corrected_rmse = (diff - diff.mean("station")).std("time")
    corrected_rmse_background = (diff_background - diff_background.mean("station")).std(
        "time"
    )
    mae_background = (np.abs(diff_background)).mean("time")
    rmse_background = np.sqrt(((diff_background) ** 2).mean("time"))
    bias = diff.mean("time")
    corr = xr.corr(measurements, model, dim="time")
    # Create DataFrame
    analysis_table = pd.DataFrame(
        {
            "mean_enhancement": mean_enhancement.values,
            "n_points": number_of_points.values,
            "rmse": rmse.values,
            "rmse_background": rmse_background.values,
            "mae": mae.values,
            "mae_background": mae_background.values,
            "corrected_rmse": corrected_rmse.values,
            "corrected_rmse_background": corrected_rmse_background.values,
            "bias": bias.values,
            "corr": corr.values,
        },
        index=mean_enhancement.station.values,
    )
    # Create summary row
    # analysis_table.loc["summary"] = {
    #     "mean_enhancement": mean_enhancement.mean().item(),
    #     "n_points": number_of_points.sum().item(),
    #     "rmse": rmse.mean().item(),
    #     "rmse_background": rmse_background.mean().item(),
    #     "mae": mae.mean().item(),
    #     "mae_background": mae_background.mean().item(),
    #     "corrected_rmse": corrected_rmse.mean().item(),
    #     "corrected_rmse_background": corrected_rmse_background.mean().item(),
    #     "bias": bias.mean().item(),
    #     "corr": corr.mean().item(),
    # }

    # Create station-average row
    # rmse = np.sqrt(((diff).mean("station") ** 2).mean("time"))
    # rmse_background = np.sqrt(((diff_background).mean("station") ** 2).mean("time"))
    # mae = np.abs((diff).mean("station")).mean("time")
    # mae_background = np.abs((diff_background).mean("station")).mean("time")
    # corr = xr.corr(measurements.mean("station"), model.mean("station"), dim="time")
    # analysis_table.loc["station_average"] = {
    #     "mean_enhancement": mean_enhancement.mean().item(),
    #     "n_points": number_of_points.mean().item(),
    #     "rmse": rmse.item(),
    #     "rmse_background": rmse_background.item(),
    #     "mae": mae.item(),
    #     "mae_background": mae_background.item(),
    #     "corrected_rmse": 0,
    #     "corrected_rmse_background": 0,
    #     "bias": bias.mean().item(),
    #     "corr": corr.item(),
    # }
    return analysis_table

In [8]:
combined, _ = p.plotting._loaders.load_combined_data()
co2 = ggp.load("co2_measurements", CONFIG).co2

INFO:root:Opening co2_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers/co2.nc
INFO:root:Opening concentration_timeseries from /Users/rmaiwald/Levante/Paris/Output/concentration_timeseries.nc


[########################################] | 100% Completed | 7.17 ss


INFO:root:Opening background_co2 from /Users/rmaiwald/Levante/Paris/Output/background_co2.nc
INFO:root:Opening co2_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers/co2.nc


In [9]:
for_table = combined.sel(station=~combined.station.str.contains("Mean"))  # Drop Mean
analysis_df = create_analysis_table(
    for_table.sel(dataset="Origins.earth"),
    for_table.sel(dataset="CO2"),
    for_table.sel(dataset="Background"),
)[["rmse", "bias"]].round(2)

In [10]:
# Calculate the available measurements in 2023 to 2024
co2_period = co2.sel(time=slice("2023", "2024"))
n_total = co2_period.sizes["time"]
counts = co2_period.count(dim="time")
co2["measurement_availability"] = (
    counts.astype(str) + " (" + (counts / n_total * 100).round(1).astype(str) + "%)"
)

# Convert to a DataFrame and select relevant columns
df = (
    co2.coords.to_dataset()
    .drop_vars("time")
    .to_pandas()[
        [
            "height",
            "latitude",
            "longitude",
            "code",
            "instrument",
            "in_gral_domain",
            "measurement_availability",
        ]
    ]
)

# Concat tables
df = pd.concat([df, analysis_df], axis="columns")

# Rename columns
df = df.rename(
    columns={
        "station": "Station",
        "height": "Height (m a.s.l.)",
        "latitude": "Latitude",
        "longitude": "Longitude",
        "code": "Code",
        "instrument": "Instrument",
        "in_gral_domain": "In GRAL Domain",
        "measurement_availability": "Number of Measurements (Percentage)",
        "rmse": "RMSE (ppm)",
        "bias": "Bias (ppm)",
    }
)

# Sort by Instrument, then by Code, and then by Height
df = df.sort_values(
    by=["In GRAL Domain", "Instrument", "Code", "Height (m a.s.l.)"],
    ascending=[False, False, True, True],
)

# Drop "Code"
df = df.drop(columns=["Code"])

# Escape underscores in the index for LaTeX
df.index = df.index.str.replace("_", "\\_", regex=False)

# Escape the percentage sign in the "Number of Measurements (Percentage)" column for LaTeX
df["Number of Measurements (Percentage)"] = df[
    "Number of Measurements (Percentage)"
].str.replace("%", "\\%", regex=False)

# Round height to the nearest meter
df["Height (m a.s.l.)"] = df["Height (m a.s.l.)"].round().astype(int)

# Round latitude and longitude to 4 decimal places
df["Latitude"] = df["Latitude"].round(4)
df["Longitude"] = df["Longitude"].round(4)


print(
    df[
        [
            "Height (m a.s.l.)",
            "Latitude",
            "Longitude",
            "Instrument",
            "In GRAL Domain",
            "Number of Measurements (Percentage)",
            "RMSE (ppm)",
            "Bias (ppm)",
        ]
    ].to_latex(float_format="%.2f")
)
df

\begin{tabular}{lrrrlrlrr}
\toprule
 & Height (m a.s.l.) & Latitude & Longitude & Instrument & In GRAL Domain & Number of Measurements (Percentage) & RMSE (ppm) & Bias (ppm) \\
\midrule
CDS\_34 & 34 & 48.90 & 2.39 & Picarro & True & 16272 (92.7\%) & 13.28 & 2.55 \\
JUS\_30 & 30 & 48.85 & 2.36 & Picarro & True & 14121 (80.5\%) & 11.95 & -0.19 \\
JUS\_40 & 40 & 48.85 & 2.36 & Picarro & True & 9758 (55.6\%) & 11.13 & 0.05 \\
MEU\_45 & 45 & 48.80 & 2.20 & Picarro & True & 11088 (63.2\%) & 5.12 & -1.83 \\
MEU\_65 & 65 & 48.80 & 2.20 & Picarro & True & 11101 (63.3\%) & 4.41 & -1.45 \\
MEU\_90 & 90 & 48.80 & 2.20 & Picarro & True & 16491 (94.0\%) & 3.34 & -0.85 \\
ROV\_103 & 103 & 48.89 & 2.42 & Picarro & True & 15623 (89.1\%) & 5.35 & -0.93 \\
BAS\_50 & 50 & 48.85 & 2.37 & K96 & True & 3929 (22.4\%) & 10.08 & -0.99 \\
BNF\_80 & 80 & 48.83 & 2.38 & K96 & True & 8872 (50.6\%) & 11.25 & -2.52 \\
BOB\_54 & 54 & 48.91 & 2.44 & K96 & True & 9872 (56.3\%) & 15.55 & -8.63 \\
CDS\_34\_K96 & 34 & 48.9

,Height (m a.s.l.),Latitude,Longitude,Instrument,In GRAL Domain,Number of Measurements (Percentage),RMSE (ppm),Bias (ppm)
CDS\_34,34,48.8956,2.3880,Picarro,True,16272 (92.7\%),13.28,2.55
JUS\_30,30,48.8464,2.3561,Picarro,True,14121 (80.5\%),11.95,-0.19
JUS\_40,40,48.8464,2.3561,Picarro,True,9758 (55.6\%),11.13,0.05
MEU\_45,45,48.8025,2.2044,Picarro,True,11088 (63.2\%),5.12,-1.83
MEU\_65,65,48.8025,2.2044,Picarro,True,11101 (63.3\%),4.41,-1.45
MEU\_90,90,48.8025,2.2044,Picarro,True,16491 (94.0\%),3.34,-0.85
ROV\_103,103,48.8854,2.4225,Picarro,True,15623 (89.1\%),5.35,-0.93
BAS\_50,50,48.8524,2.3704,K96,True,3929 (22.4\%),10.08,-0.99
BNF\_80,80,48.8334,2.3774,K96,True,8872 (50.6\%),11.25,-2.52
BOB\_54,54,48.9077,2.4445,K96,True,9872 (56.3\%),15.55,-8.63


# Ensemble size


In [11]:
conc_series = ggp.load("concentration_timeseries", CONFIG)
loss_diff = conc_series["loss_diff"].sel(loss_type="rmse - filter: True")
mask = (loss_diff < 0.1).sum("best_sim_id")
print(
    "Take the 95% percentile as a threshold for the number of simulations that are "
    "below the 0.1 RMSE threshold."
)
display(mask.to_pandas().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
limit = int((0.9 * len(mask)))
print(f"Select top 10% of the simulations: {len(mask) - limit}/{len(mask)}")
mean = mask.sortby(mask).values[limit:].mean()
print(
    f"Mean number of simulations below the 0.1 RMSE threshold for the top 10%: "
    f"{mean:.2f}"
)

INFO:root:Opening concentration_timeseries from /Users/rmaiwald/Levante/Paris/Output/concentration_timeseries.nc


Take the 95% percentile as a threshold for the number of simulations that are below the 0.1 RMSE threshold.


count    17544.000000
mean         3.912677
std          4.341492
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
90%          7.000000
95%         10.000000
99%         24.000000
max         85.000000
Name: loss_diff, dtype: float64

Select top 10% of the simulations: 1755/17544
Mean number of simulations below the 0.1 RMSE threshold for the top 10%: 13.45


# Wind Measurements


In [12]:
model_meteo_timeseries = (
    ggp.load("model_meteo_timeseries", CONFIG)
    .sel(loss_type="rmse - filter: True")
    .load()
)
meteo = ggp.load("meteo_measurements", CONFIG)
meteo = meteo.sel(
    station=model_meteo_timeseries.station, time=model_meteo_timeseries.time
).load()

INFO:root:Opening gramm_meteo_timeseries from /Users/rmaiwald/Levante/Paris/Output/gramm_meteo_timeseries.nc
INFO:root:Opening gral_meteo_timeseries from /Users/rmaiwald/Levante/Paris/Output/gral_meteo_timeseries.nc
INFO:root:Selecting model data for station LONGCHAMP from model gral
INFO:root:Selecting model data for station PARIS-MONTSOURIS from model gral
INFO:root:Selecting model data for station TOUR EIFFEL from model gramm
INFO:root:Selecting model data for station LFPB from model gramm
INFO:root:Selecting model data for station LFPN_2 from model gramm
INFO:root:Selecting model data for station LFPO from model gramm
INFO:root:Selecting model data for station LFPV from model gramm
INFO:root:Selecting model data for station Cité des Sciences from model gral
INFO:root:Selecting model data for station Meudon from model gramm
INFO:root:Selecting model data for station Romainville from model gral
INFO:root:Opening meteo_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measuremen

In [13]:
wind_speed_diff = model_meteo_timeseries["wind_speed"] - meteo["wind_speed"]
wind_direction_diff = ggp.processing.circular_diff(
    model_meteo_timeseries["wind_direction"], meteo["wind_direction"]
)
u_wind_diff = model_meteo_timeseries["u"] - meteo["u_wind"]
v_wind_diff = model_meteo_timeseries["v"] - meteo["v_wind"]

In [14]:
def model_performance_df(
    model: xr.Dataset,
    meteo: xr.Dataset,
) -> pd.DataFrame:
    """Create a DataFrame of model performance metrics from wind model and measurements.

    Computes bias, MAE, RMSE, and Pearson R per station averaged over time and
    (if present) all ensemble members (``best_sim_id`` dimension). u and v are
    combined into a single wind vector variable: bias is reported as separate u/v
    columns, MAE and RMSE are computed on the vector magnitude (no R for the vector).
    An additional column reports the mean observed wind speed per station.

    Parameters
    ----------
    model : xr.Dataset
        Model output with variables ``wind_speed``, ``wind_direction``, ``u``, ``v``.
    meteo : xr.Dataset
        Measurements with variables ``wind_speed``, ``wind_direction``,
        ``u_wind``, ``v_wind``.

    Returns
    -------
    pd.DataFrame
        Multi-level column DataFrame indexed by station with columns
        ``(variable, metric)``, plus a final "Mean" row.
    """

    def _avg_dims(da: xr.DataArray) -> list[str]:
        return [d for d in da.dims if d != "station"]

    def _metrics(model_var: xr.DataArray, meas_var: xr.DataArray) -> pd.DataFrame:
        diff: xr.DataArray = (model_var - meas_var).astype(float)  # type: ignore[assignment]
        avg_dims = _avg_dims(diff)
        bias = diff.mean(dim=avg_dims)
        mae = abs(diff).mean(dim=avg_dims)
        rmse = (diff**2).mean(dim=avg_dims) ** 0.5
        r = xr.corr(model_var.astype(float), meas_var.astype(float), dim=avg_dims)  # type: ignore[arg-type]
        return pd.DataFrame(
            {
                "Bias (m/s)": bias.to_pandas(),
                # "MAE": mae.to_pandas(),
                "RMSE (m/s)": rmse.to_pandas(),
                "R": r.to_pandas(),
            }
        )

    def _vector_metrics(
        u_model: xr.DataArray,
        v_model: xr.DataArray,
        u_meas: xr.DataArray,
        v_meas: xr.DataArray,
    ) -> pd.DataFrame:
        fu: xr.DataArray = (u_model - u_meas).astype(float)  # type: ignore[assignment]
        fv: xr.DataArray = (v_model - v_meas).astype(float)  # type: ignore[assignment]
        avg_dims = _avg_dims(fu)
        u_bias = fu.mean(dim=avg_dims)
        v_bias = fv.mean(dim=avg_dims)
        mae = ((fu**2 + fv**2) ** 0.5).mean(dim=avg_dims)
        rmse = ((fu**2 + fv**2).mean(dim=avg_dims)) ** 0.5
        return pd.DataFrame(
            {
                # "Bias (u)": u_bias.to_pandas(),
                # "Bias (v)": v_bias.to_pandas(),
                # "MAE": mae.to_pandas(),
                "RMSE (m/s)": rmse.to_pandas(),
            }
        )

    wind_direction_diff = ggp.processing.circular_diff(
        model["wind_direction"], meteo["wind_direction"]
    )
    wind_dir_corr = xr.corr(
        model["wind_direction"].astype(float),  # type: ignore[arg-type]
        meteo["wind_direction"].astype(float),  # type: ignore[arg-type]
        dim=_avg_dims(wind_direction_diff),
    )

    obs_speed = (
        meteo["wind_speed"]
        .astype(float)
        .mean(dim=_avg_dims(meteo["wind_speed"]))  # type: ignore[assignment]
    )

    frames: dict[str, pd.DataFrame] = {
        "Obs. speed": pd.DataFrame({"(m/s)": obs_speed.to_pandas()}),
        "Vector": _vector_metrics(
            model["u"], model["v"], meteo["u_wind"], meteo["v_wind"]
        ),
        "Speed": _metrics(model["wind_speed"], meteo["wind_speed"]),
        "Dir. (°)": pd.DataFrame(
            {
                # "Bias": wind_direction_diff.astype(float).mean(dim=_avg_dims(wind_direction_diff)).to_pandas(),  # type: ignore[assignment]
                # "MAE": abs(wind_direction_diff.astype(float)).mean(dim=_avg_dims(wind_direction_diff)).to_pandas(),  # type: ignore[assignment]
                "RMSE": ((wind_direction_diff.astype(float) ** 2).mean(dim=_avg_dims(wind_direction_diff)) ** 0.5).to_pandas(),  # type: ignore[assignment]
                # "R": wind_dir_corr.to_pandas(),
            }
        ),
    }

    df = pd.concat(frames, axis=1)

    mean_row = df.mean().rename("Mean").to_frame().T
    df = pd.concat([df, mean_row])
    df.index.name = "Station"
    return df


df = model_performance_df(model_meteo_timeseries, meteo).round(2)

# Replace all underscores in the index with escaped underscores for LaTeX
df.index = df.index.str.replace("_", "\\_", regex=False)

In [15]:
latex = df.to_latex(float_format="%.2f")

# Insert \midrule before the last data row and make it bold
lines = latex.splitlines()
# Find the last data line (before \bottomrule)
for i in range(len(lines) - 1, -1, -1):
    if lines[i].strip() == r"\bottomrule":
        # Insert \midrule before the last data row
        last_data_idx = i - 1
        cells = lines[last_data_idx].rstrip(r" \\").split(" & ")
        lines[last_data_idx] = (
            " & ".join(f"\\textbf{{{c.strip()}}}" for c in cells) + r" \\"
        )
        lines.insert(last_data_idx, r"\midrule")
        break

print("\n".join(lines))
display(df.style.background_gradient(axis="index").format("{:.2f}"))

\begin{tabular}{lrrrrrr}
\toprule
 & Obs. speed & Vector & \multicolumn{3}{r}{Speed} & Dir. (°) \\
 & (m/s) & RMSE (m/s) & Bias (m/s) & RMSE (m/s) & R & RMSE \\
Station &  &  &  &  &  &  \\
\midrule
LONGCHAMP & 2.51 & 1.39 & 0.25 & 1.13 & 0.75 & 46.44 \\
PARIS-MONTSOURIS & 3.11 & 1.63 & -0.97 & 1.37 & 0.74 & 29.57 \\
TOUR EIFFEL & 7.20 & 2.65 & -0.56 & 2.36 & 0.80 & 19.57 \\
LFPB & 3.59 & 2.20 & -0.70 & 1.45 & 0.78 & 49.23 \\
LFPN\_2 & 3.83 & 1.81 & -0.66 & 1.40 & 0.83 & 40.45 \\
LFPO & 3.73 & 1.78 & -0.49 & 1.30 & 0.80 & 37.96 \\
LFPV & 3.71 & 1.79 & -1.00 & 1.46 & 0.81 & 34.67 \\
Cité des Sciences & 2.64 & 1.60 & 0.63 & 1.16 & 0.80 & 29.56 \\
Meudon & 4.72 & 3.34 & -0.92 & 1.79 & 0.74 & 41.01 \\
Romainville & 6.03 & 3.22 & -0.64 & 2.02 & 0.75 & 27.87 \\
\midrule
\textbf{Mean} & \textbf{4.11} & \textbf{2.14} & \textbf{-0.51} & \textbf{1.54} & \textbf{0.78} & \textbf{35.64} \\
\bottomrule
\end{tabular}
